# Matchmaking System

We want to design a new matchmaking system for the AI Arena StarCraft 2 bot ladder **without rounds** and **without divisions** (divisions may still be present in the frontend to report progress, but are not used for matchmaking itself).

The matchmaking must ensure that **all** bots

- play a "fair" number of matches;
- play predominantly matches against opponents of a similar skill;
- play matches against varied opponents within those of similar skill;
- play matches which are somewhat evenly distributed in time.

We verify these requirements by running the matchmaker on a [model of the real ladder](ladder_model.ipynb) and checking the match frequency distributions, rating-difference distributions, and opponent distributions.

## Scoring Function

The matchmaker is **greedy** with a stochastic relaxation: whenever a game slot becomes available, it scores all pairs of idle bots and samples a match using a softmax distribution over the scores. The score for a candidate match between bots A and B is

$$
\text{score}(A, B) = w_\text{skill} \cdot f_\text{skill}(A, B) + w_\text{fair} \cdot f_\text{fair}(A, B) + w_\text{var} \cdot f_\text{var}(A, B),
$$

where the components correspond to three of the four matchmaking requirements; the time-distribution requirement is handled implicitly, since a bot cannot accumulate idle time without also accumulating game-count deficit, which is captured by the fairness term.

### Single-Instance Constraint

Before scoring, the matchmaker applies a hard filter: any bot already in an active match is excluded from the candidate pool, regardless of whether the bot has `bot_data_enabled` set. AI Arena itself only *physically* requires single-instance execution for data-enabled bots (non-data bots have no persistent state between matches and may run in parallel), but this matchmaker deliberately treats every bot as single-instance. The motivation:

- Scoring decisions depend on state that only updates when a match *completes* — rating (for skill), games-in-last-24h (for fairness), and games-since-last-opponent (for variety). Dispatching a second match for a bot before its first finishes scores the second decision against stale information and inflates the bot's apparent underplayed-ness until the in-flight matches land.
- Non-data bots would otherwise be handed streaks of concurrent matches by the fairness term — they'd accumulate "deficit" quickly because completed-game counts lag dispatched-game counts, which distorts the measured fairness signal.

Applying the constraint uniformly keeps every decision aligned with fully-applied state at the cost of slightly reduced throughput for non-data bots.

### Rematch Prevention

Before scoring, the matchmaker applies a hard constraint: a pair $(A, B)$ is excluded if either bot's most recent completed match was against the other. This prevents back-to-back rematches regardless of score, and forces a bot to have at least one intermediate match before it can face the same opponent again. The constraint is asymmetric: after $A$ plays $B$, then $A$ plays $C$, the pair $(A, B)$ is still blocked until $B$ has also played someone else. This avoids a structural failure mode in which two isolated top bots accumulate synchronized fairness debt and repeatedly pair on that alone.

### Skill Matching

Matches between similarly-ranked bots should be preferred. Rather than penalising the raw rating difference, we penalise the **rank** difference, using a Cauchy (Lorentzian) kernel:

$$
f_\text{skill}(A, B) = \frac{1}{1 + \left(\dfrac{\text{rank}_A - \text{rank}_B}{\tau}\right)^2},
$$

where ranks are assigned by current rating (best = 0) and $\tau$ is measured in rank positions.

**Why ranks, not ratings.** This makes the skill tolerance *adaptive to ladder density*: in dense parts of the ladder, adjacent ranks mean small rating gaps; in sparse parts (e.g. an orphan bot at the bottom with no near-ELO neighbours), adjacent ranks may span a large rating gap, but that is exactly as it should be — there is nothing closer to play. A rating-based penalty would instead force the orphan to repeatedly play its single nearest neighbour.

**Why Cauchy, not Gaussian or exponential.** We need three properties: (i) a *flat* top so that adjacent ranks (1, 2, 3, ...) are treated as roughly equivalent and the top bot is not locked into always playing rank 2; (ii) a *smooth* transition (no cusps) so there are no artificial preferences for single-rank differences; (iii) a *heavy tail* so that the mid-range (rank-diff 10–30) retains discriminative signal against cross-ladder pairings. A Gaussian is flat-topped and smooth but its tail decays too quickly, leaving the far-range skill score near zero and allowing variety to dominate cross-ladder matches. An exponential has a heavier tail but has a cusp at zero, giving an artificial preference for adjacent ranks over rank-diff 2 or 3. The Cauchy kernel has all three properties: $1/(1 + x^2)$ decay with smooth, flat behaviour near zero. Values in $[0, 1]$.

### Fairness

Bots that have played fewer games should be prioritised. Let $g_i$ be the number of games bot $i$ has completed in the last 24 hours and $\bar{g}$ the mean across all currently active bots. Define each bot's normalised positive deficit $d_i = \max(\bar{g} - g_i, 0) / \bar{g}$, and aggregate with an RMS:

$$
f_\text{fair}(A, B) = \sqrt{\frac{d_A^2 + d_B^2}{2}}.
$$

Clipping at zero drops the "overplayed" side: once a bot is above average, it no longer *subtracts* from the fairness score, but it also does not boost pairings. The RMS rewards pairings where *both* bots are underplayed more than pairings where only one is. Dividing by $\bar{g}$ normalises the deficit to a fraction, keeping the component on a comparable scale to the others. Bots that recently joined the ladder naturally receive a temporary priority boost since their game count in the window is low. Values in $[0, 1]$.

### Opponent Variety

Repeated match-ups should be discouraged. Let $a_{AB}$ be the number of games $A$ has played since last facing $B$, and $b_{AB}$ the same for $B$ ($a_{AB} = b_{AB} = \infty$ if they have never met):

$$
f_\text{var}(A, B) = 1 - \exp\!\left(-\frac{\sqrt{a_{AB} \cdot b_{AB}}}{\lambda}\right),
$$

where $\lambda$ controls how quickly a past match-up is "forgotten". The geometric mean $\sqrt{a_{AB} \cdot b_{AB}}$ gives partial credit when one bot has diversified a lot but the other hasn't, while still requiring both to have moved on. Values in $[0, 1]$.

### Softmax Sampling

Rather than picking the single highest-scoring pair, the matchmaker samples a pair from a softmax distribution over scores:

$$
P(A, B) \propto \exp\!\left(\frac{\text{score}(A, B)}{T}\right),
$$

where $T$ is the sampling temperature. A pure argmax is recovered as $T \to 0$; larger $T$ introduces more randomness. This serves two purposes:

1. **Smoother response to weight changes.** With pure argmax, the matchmaker applies `argmax` across $\sim 3000$ candidate pairs per decision; the winner is whoever edges out by any margin, no matter how small. A small change in the weights can deterministically flip which *category* of pair (near-rank / cross-ladder / underplayed) wins consistently, and that flip is then amplified across many thousands of matches. Softmax sampling lets near-ties be decided by chance, dampening the winner-take-all amplification.
2. **Beneficial stochasticity.** A deterministic matchmaker would produce the same matches every run given the same state; real ladders have natural variation. Softmax gives each near-optimal pairing a chance to occur, which broadens opponent variety without requiring explicit variety pressure.

The temperature should be small relative to the weighted score values — large enough that near-ties ($\Delta\text{score} \lesssim T$) randomise meaningfully, but not so large that saturated far-rank pairs (high variety, low skill) compete with near-rank pairs.

### Parameters

| Parameter | Role |
|-----------|------|
| $w_\text{skill},\, w_\text{fair},\, w_\text{var}$ | Relative importance of each objective |
| $\tau$ | Rank-difference tolerance (in rank positions) |
| $\lambda$ | Opponent-variety decay rate |
| $T$ | Softmax sampling temperature |

Since the components have different scales, the weights absorb normalisation — they are not directly comparable across components.